In [ ]:
!pip install pyiceberg[s3fs,pandas,pyarrow] boto3

In [ ]:
import os
from pyiceberg.catalog import load_catalog
import pandas as pd
import pyarrow as pa
from datetime import datetime

os.environ['PYICEBERG_DOWNCAST_NS_TIMESTAMP_TO_US_ON_WRITE'] = 'true'

In [ ]:
# Connect to Iceberg REST Catalog
catalog = load_catalog(
    "rest",
    **{
        "uri": "http://iceberg-rest:8181",
        "s3.endpoint": "http://minio:9000",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "Password!",
        "s3.path-style-access": "true",
        "s3.region": "us-east-1" 
    }
)

print("Connected to Iceberg REST Catalog!")
print(f"Catalog properties: {catalog.properties}")

In [ ]:
# Create Namespace (Database)
try:
    catalog.create_namespace("demo")
    print("Created namespace: demo")
except Exception as e:
    print(f"Namespace may already exist: {e}")

# List namespaces
print("\nAvailable namespaces:", catalog.list_namespaces())

In [ ]:
# Drop existing table
try:
    catalog.drop_table("demo.events")
    print("Dropped existing table")
except:
    pass

In [ ]:
# Create Table Schema
from pyiceberg.schema import Schema
from pyiceberg.types import (
    NestedField,
    StringType,
    DoubleType,
    TimestamptzType, # NOTE, timeplus use timestamptz for timestamp
    LongType
)

schema = Schema(
    NestedField(1, "id", LongType(), required=False),  # ← Changed to LongType and optional
    NestedField(2, "timestamp", TimestamptzType(), required=False),  # ← Made optional
    NestedField(3, "user_id", StringType(), required=False),  # ← Made optional
    NestedField(4, "event_type", StringType(), required=False),  # ← Made optional
    NestedField(5, "value", DoubleType(), required=False),
)

# Create table
try:
    table = catalog.create_table(
        identifier="demo.events",
        schema=schema,
    )
    print("Created table: demo.events")
except Exception as e:
    print(f"Table may already exist: {e}")
    table = catalog.load_table("demo.events")

print(f"\nTable schema:\n{table.schema()}")

In [ ]:
# Write Data

data = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "timestamp": pd.date_range("2024-01-01", periods=5, freq="H", tz='UTC'),
    "user_id": ["user_1", "user_2", "user_1", "user_3", "user_2"],
    "event_type": ["login", "click", "purchase", "login", "click"],
    "value": [None, 10.5, 99.99, None, 25.0]
})

# Convert to PyArrow table (PyArrow will handle the precision automatically)
arrow_table = pa.Table.from_pandas(data)
print("Writing data to Iceberg table...")
table.append(arrow_table)
print("✓ Data written successfully!")

In [ ]:
#  Read Data

from pyiceberg.table import TableProperties
table = catalog.load_table("demo.events")

with table.transaction() as txn:
    txn.set_properties(
        **{TableProperties.DEFAULT_NAME_MAPPING: table.metadata.schema().name_mapping.model_dump_json()}
    )

print(f"name-mapping: {table.metadata.name_mapping()}")

print("Reading data from table...")
df = table.scan().to_pandas()
print(f"\nTable has {len(df)} rows:\n")
print(df)

In [ ]:
# Query with Filters
print("\n--- Filtering: event_type = 'login' ---")
df_filtered = table.scan(
    row_filter="event_type == 'login'"
).to_pandas()
print(df_filtered)

print("\n--- Filtering: value > 20 ---")
df_filtered2 = table.scan(
    row_filter="value > 20"
).to_pandas()
print(df_filtered2)

In [ ]:
# Append More Data
new_data = pd.DataFrame({
    "id": [6, 7, 8],
    "timestamp": pd.date_range("2024-01-01 05:00:00", periods=3, freq="H", tz='UTC'),
    "user_id": ["user_1", "user_4", "user_2"],
    "event_type": ["logout", "login", "purchase"],
    "value": [None, None, 149.99]
})


# Convert to PyArrow table
arrow_new_data = pa.Table.from_pandas(new_data)

print("Appending more data...")
table.append(arrow_new_data)
print("✓ Data appended!")

# Read updated data
df_updated = table.scan().to_pandas()
print(f"\nTable now has {len(df_updated)} rows")
print(df_updated)

In [ ]:
# Table History & Time Travel
print("--- Table History ---")
snapshots = table.metadata.snapshots
for snapshot in snapshots:
    print(f"Snapshot ID: {snapshot.snapshot_id}, Timestamp: {snapshot.timestamp_ms}")

# Time travel - read data as of first snapshot
if len(snapshots) >= 2:
    first_snapshot_id = snapshots[0].snapshot_id
    print(f"\n--- Time Travel to Snapshot {first_snapshot_id} ---")
    df_historical = table.scan(snapshot_id=first_snapshot_id).to_pandas()
    print(f"Historical data ({len(df_historical)} rows):")
    print(df_historical)